# DPO Fine-Tuning with Intel Orca Dataset on Microsoft Foundry

This notebook demonstrates how to fine-tune language models using **Direct Preference Optimization (DPO)** with the Intel Orca DPO Pairs dataset.

## What You'll Learn
1. Understand DPO fine-tuning
2. Prepare and format DPO training data  
3. Upload datasets to Microsoft Foundry
4. Create and monitor a DPO fine-tuning job
5. Evaluate your fine-tuned model

Note: Execute each cell in sequence.

## 1. Setup and Installation

Install all required packages from requirements.txt

In [1]:
# Dependencies are installed in the isolated Python 3.12 environment before execution.
import sys
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")


Python: 3.12.13 (main, Jul 18 2026, 17:08:38) [MSC v.1944 64 bit (AMD64)]
Executable: C:\Data\AIP\fine-tuning\Demos\DPO_Intel_Orca\.venv\Scripts\python.exe


## 2. Import Libraries

In [2]:
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from openai import APIConnectionError, APITimeoutError

print("All libraries imported successfully")


All libraries imported successfully


## 3. Define Evaluation Function

Function to evaluate model performance using Azure AI Evaluation SDK.

In [3]:
def evaluate_model(deployment_name, num_samples=5, evaluator_model=None):
    from azure.ai.evaluation import evaluate, CoherenceEvaluator, FluencyEvaluator, GroundednessEvaluator

    evaluator_model = evaluator_model or os.getenv("DEPLOYMENT_NAME", "gpt-4.1-mini")
    print(f"Evaluating deployment: {deployment_name}")
    print(f"Using evaluator model: {evaluator_model}")
    print(f"Using {num_samples} validation samples")

    eval_data = []
    with open("validation.jsonl", "r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if index >= num_samples:
                break
            sample = json.loads(line)
            messages = sample["input"]["messages"]
            query = next(message["content"] for message in messages if message["role"] == "user")
            response = openai_client.chat.completions.create(model=deployment_name, messages=messages)
            ground_truth = next(
                message["content"] for message in sample["preferred_output"] if message["role"] == "assistant"
            )
            eval_data.append(
                {
                    "query": query,
                    "response": response.choices[0].message.content,
                    "ground_truth": ground_truth,
                }
            )
            print(f"Processed {index + 1}/{num_samples}")

    eval_file = f"evaluation_data_{deployment_name.replace('-', '_').replace('.', '_')}.jsonl"
    with open(eval_file, "w", encoding="utf-8") as handle:
        for item in eval_data:
            handle.write(json.dumps(item) + "\n")

    model_config = {
        "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT", endpoint.split(".services.ai.azure.com")[0] + ".openai.azure.com"),
        "azure_deployment": evaluator_model,
        "api_version": "2025-04-01-preview",
    }
    evaluators = {
        "coherence": CoherenceEvaluator(model_config=model_config, credential=credential),
        "fluency": FluencyEvaluator(model_config=model_config, credential=credential),
        "groundedness": GroundednessEvaluator(model_config=model_config, credential=credential),
    }
    results = evaluate(
        data=eval_file,
        evaluators=evaluators,
        evaluator_config={
            "default": {
                "column_mapping": {
                    "query": "${data.query}",
                    "response": "${data.response}",
                    "ground_truth": "${data.ground_truth}",
                }
            },
            "groundedness": {
                "column_mapping": {
                    "query": "${data.query}",
                    "response": "${data.response}",
                    "context": "${data.ground_truth}",
                }
            },
        },
        output_path=f"./evaluation_results_{deployment_name.replace('-', '_').replace('.', '_')}",
    )
    print(json.dumps(results.get("metrics", {}), indent=2, default=str))
    return results


## 4. Configure Azure Environment
Set your Microsoft Foundry Project endpoint and model name. We're using **gpt-4.1-mini** in this example, but you can use other supported GPT models. Copy the file `.env.template` (located in this folder), and save it as file named `.env`. Enter appropriate values for the environment variables used for the job you want to run. 

```
# Required for DPO Fine-Tuning
MICROSOFT_FOUNDRY_PROJECT_ENDPOINT=<your-endpoint> 
AZURE_SUBSCRIPTION_ID=<your-subscription-id>
AZURE_RESOURCE_GROUP=<your-resource-group>
AZURE_AOAI_ACCOUNT=<your-foundry-account-name>
MODEL_NAME=<your-base-model-name>

# Required for Model Local Evaluation
AZURE_OPENAI_ENDPOINT=<your-azure-openai-endpoint>
AZURE_OPENAI_KEY=<your-azure-openai-api-key>
DEPLOYMENT_NAME=<your-deployment-name>
```

In [4]:
load_dotenv()

endpoint = os.getenv(
    "MICROSOFT_FOUNDRY_PROJECT_ENDPOINT",
    "https://swec-prakharg-demo-2026.services.ai.azure.com/api/projects/swec-prakharg-demo-2026",
)
model_name = os.getenv("MODEL_NAME", "gpt-4o-mini-2024-07-18")
def write_subset(source_name, destination_name, limit):
    lines = Path(source_name).read_text(encoding="utf-8").splitlines()
    Path(destination_name).write_text("\n".join(lines[:limit]) + "\n", encoding="utf-8")
    return min(limit, len(lines))

training_file_path = "training_run.jsonl"
validation_file_path = "validation_run.jsonl"
training_count = write_subset("training.jsonl", training_file_path, 25)
validation_count = write_subset("validation.jsonl", validation_file_path, 10)

print(f"Project endpoint: {endpoint}")
print(f"Base model: {model_name}")
print(f"Run dataset: {training_count} training examples, {validation_count} validation examples")


Project endpoint: https://eastus2-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus2-prakharg-demo-2026
Base model: gpt-4o-mini-2024-07-18
Run dataset: 25 training examples, 10 validation examples


## 4. Connect to Microsoft Foundry Project

Connect to Microsoft Foundry Project using Azure credential authentication. This initializes the project client and OpenAI client needed for fine-tuning workflows. Ensure you have the **Azure AI User** role assigned to your account for the Microsoft Foundry Project resource.

In [5]:
credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()
deployments = [deployment.name for deployment in project_client.deployments.list()]
print(f"Connected to Microsoft Foundry Project; deployments: {deployments}")


Connected to Microsoft Foundry Project; deployments: ['gpt-5.4-mini', 'image-breed-gpt-41-mini-base', 'gpt-audio', 'gpt-4o-mini', 'cnn-dailymail-1ea1d3cf7f92', 'intel-orca-dpo-ac5556a1', 'nl2py-eus2-32605705a8']


## 5. Upload Training Files

Upload the training and validation JSONL files to Microsoft Foundry. Each file is assigned a unique ID that will be referenced when creating the fine-tuning job.

In [6]:
print("Uploading training file...")
with open(training_file_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f" Training file ID: {train_file.id}")

print("\nUploading validation file...")
with open(validation_file_path, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f" Validation file ID: {validation_file.id}")

Uploading training file...


 Training file ID: file-ee9e5cd5c7ac4ebc9fd2c23be7c40d69

Uploading validation file...


 Validation file ID: file-a70bb4b325e8456dafa746b310e08568


In [7]:
print("Waiting for files to be processed...")
openai_client.files.wait_for_processing(train_file.id)
openai_client.files.wait_for_processing(validation_file.id)
print(" Files ready!")

Waiting for files to be processed...


 Files ready!


## 7. Evaluate Base Model

Establish baseline performance metrics by evaluating the base model before DPO fine-tuning. This provides a comparison point to measure improvements after training.



In [8]:
base_deployment = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-mini")
print(f"Evaluating base model: {base_deployment}")
base_results = evaluate_model(base_deployment, num_samples=3)


Evaluating base model: gpt-4o-mini


Evaluating deployment: gpt-4o-mini
Using evaluator model: gpt-4o-mini
Using 3 validation samples


Processed 1/3


Processed 2/3


Processed 3/3


...

======= Run Summary =======

Run name: "fluency_20261003_064802_735827"
Run status: "Completed"
Start time: "2026-10-03 06:48:02.735827+00:00"
Duration: "0:00:12.614245"



2026-10-02 23:48:48 -0700   63628 execution.bulk     INFO     Finished 3 / 3 lines.


2026-10-02 23:48:48 -0700   63628 execution.bulk     INFO     Average execution time for completed lines: 15.34 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "groundedness_20261003_064802_735827"
Run status: "Completed"
Start time: "2026-10-03 06:48:02.735827+00:00"
Duration: "0:00:46.834974"

======= Combined Run Summary (Per Evaluator) =======

{
    "coherence": {
        "status": "Completed",
        "duration": "0:00:11.568246",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "fluency": {
        "status": "Completed",
        "duration": "0:00:12.614245",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    },
    "groundedness": {
        "status": "Completed",
        "duration": "0:00:46.834974",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,


## 8. Create DPO Fine-Tuning Job
Create a DPO fine-tuning job with your uploaded datasets. Configure the following hyperparameters to control the training process:

1. n_epochs (3): Number of complete passes through the training dataset. More epochs can improve performance but may lead to overfitting. Typical range: 1-10.
2. batch_size (1): Number of training examples processed together in each iteration. Smaller batches (1-2) are common for DPO to maintain training stability.
3. learning_rate_multiplier (1.0): Scales the default learning rate. Values < 1.0 make training more conservative, while values > 1.0 speed up learning but may cause instability. Typical range: 0.1-2.0.
Adjust these values based on your dataset size and desired model behavior. 

Start with these defaults and experiment if needed.

In [9]:
resume_job_id = os.getenv("FINE_TUNING_JOB_ID")
if resume_job_id:
    fine_tuning_job = openai_client.fine_tuning.jobs.retrieve(resume_job_id)
    print(f"Resuming existing job after a local execution interruption: {resume_job_id}")
else:
    fine_tuning_job = openai_client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        model=model_name,
        method={
            "type": "dpo",
            "dpo": {
                "hyperparameters": {
                    "n_epochs": 1,
                    "batch_size": 1,
                    "learning_rate_multiplier": 1.0,
                }
            },
        },
        suffix="intel-orca-dpo-4omini",
        extra_body={"trainingType": "GlobalStandard"},
    )
print(f"Job ID: {fine_tuning_job.id}")
print(f"Status: {fine_tuning_job.status}")


## 9. Monitor Training Progress
Check the status of your fine-tuning job and track progress. You can view the current status, and recent training events. Training duration varies based on dataset size, model, and hyperparameters - typically ranging from minutes to several hours.

In [10]:
terminal_states = {"succeeded", "failed", "cancelled"}
seen_events = set()
consecutive_connection_errors = 0
while True:
    try:
        job_status = openai_client.fine_tuning.jobs.retrieve(fine_tuning_job.id)
        consecutive_connection_errors = 0
    except (APIConnectionError, APITimeoutError) as error:
        consecutive_connection_errors += 1
        print(f"Transient monitoring error {consecutive_connection_errors}/10: {error}")
        if consecutive_connection_errors >= 10:
            raise
        time.sleep(60)
        continue
    print(f"{time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())} status={job_status.status}")
    for event in reversed(list(openai_client.fine_tuning.jobs.list_events(fine_tuning_job.id, limit=20))):
        event_key = (getattr(event, "id", None), getattr(event, "message", None))
        if event_key not in seen_events:
            seen_events.add(event_key)
            print(f"  {getattr(event, 'message', event)}")
    if job_status.status in terminal_states:
        break
    time.sleep(60)

if job_status.status != "succeeded":
    print(f"Fine-tuning job {fine_tuning_job.id} ended with status {job_status.status}: {job_status.error}")


In [11]:
events = list(openai_client.fine_tuning.jobs.list_events(fine_tuning_job.id, limit=100))
for event in reversed(events):
    print(event.message)


## 10. Retrieve Fine-Tuned Model
After the fine-tuning job succeeded, retrieve the fine-tuned model ID. This ID is required to make inference calls with your customized model.

In [12]:
completed_job = openai_client.fine_tuning.jobs.retrieve(fine_tuning_job.id)
fine_tuned_model_id = completed_job.fine_tuned_model
if fine_tuned_model_id:
    print(f"Fine-tuned Model ID: {fine_tuned_model_id}")
else:
    print(f"No fine-tuned model was produced: {completed_job.error}")


## 11. Deploy the fine-tuned Model

Deploy the fine-tuned model to Azure OpenAI as a deployment endpoint. This step is required before making inference calls. The deployment uses GlobalStandard SKU with 50 capacity.

In [13]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID", "ba7979f7-d040-49c9-af1a-7414402bf622")
resource_group = os.getenv("AZURE_RESOURCE_GROUP", "prakharg-demo-2026")
account_name = os.getenv("AZURE_AOAI_ACCOUNT", "swec-prakharg-demo-2026")
deployment_name = f"intel-orca-dpo-{fine_tuning_job.id[-8:]}"

if fine_tuned_model_id and subscription_id and resource_group and account_name:
    with CognitiveServicesManagementClient(credential=credential, subscription_id=subscription_id) as cogsvc_client:
        deployment_config = Deployment(
            properties=DeploymentProperties(
                model=DeploymentModel(format="OpenAI", name=fine_tuned_model_id, version="1")
            ),
            sku=Sku(name="GlobalStandard", capacity=50),
        )
        poller = cogsvc_client.deployments.begin_create_or_update(
            resource_group_name=resource_group,
            account_name=account_name,
            deployment_name=deployment_name,
            deployment=deployment_config,
        )
        poller.result()
    print(f"Model deployment completed: {deployment_name}")
else:
    deployment_name = None
    print("Deployment skipped: no fine-tuned model is available.")


## 12. Test Your Fine-Tuned Model

Validate your fine-tuned model by running test inferences. This helps you assess whether the DPO training successfully aligned the model with your preferred response patterns from the training data

In [14]:
if deployment_name:
    print(f"Testing fine-tuned model via deployment: {deployment_name}")
    response = openai_client.chat.completions.create(
        model=deployment_name,
        messages=[{"role": "user", "content": "Explain machine learning in simple terms."}],
    )
    print(f"Model response: {response.choices[0].message.content}")
else:
    print("Inference skipped because the fine-tuned model could not be deployed.")


## 13. Evaluate Fine-Tuned Model

Evaluate your model using Azure AI Evaluation SDK to measure quality improvements from DPO fine-tuning.

We'll assess 3 key metrics:
- **Coherence**: Logical flow and structure
- **Fluency**: Grammatical correctness and naturalness
- **Groundedness**: Factual accuracy against context

In [15]:
if deployment_name:
    print(f"Evaluating fine-tuned model: {deployment_name}")
    finetuned_results = evaluate_model(
        deployment_name, num_samples=3, evaluator_model=os.getenv("DEPLOYMENT_NAME", "gpt-4.1-mini")
    )
else:
    finetuned_results = {"skipped": "Fine-tuned deployment unavailable"}
    print("Fine-tuned evaluation skipped because the deployment is unavailable.")


## 13.1 Model Comparison Results

Below is an example comparison between base model and fine-tuned model evaluation results (using 50 samples):

| Metric | Base Model | Fine-Tuned Model | Change | Status |
|--------|-----------|------------------|--------|--------|
| **Coherence** | 4.3000 | 3.8400 | -0.4600 | Decreased |
| **Fluency** | 3.5800 | 2.7000 | -0.8800 | Decreased |
| **Groundedness** | 4.1000 | 3.1000 | -1.0000 | Decreased |

### Understanding the Results

The example above shows that the fine-tuned model performed worse than the base model across all metrics. This indicates that the DPO training did not improve model quality with the current configuration.

### How to Improve Results

To achieve better fine-tuned model performance, experiment with:

**1. Hyperparameter Tuning:**
- **Reduce epochs**: Try `n_epochs=1` or `n_epochs=2` to prevent overfitting
- **Lower learning rate**: Set `learning_rate_multiplier=0.5` or `0.1` for more conservative training
- **Adjust batch size**: Keep at 1-2 for DPO stability

**2. Training Data:**
- **Increase sample size**: Use more training examples (e.g., 100-1000 samples)
- **Verify data quality**: Ensure "preferred_output" responses are truly higher quality than rejected ones
- **Review data format**: Confirm DPO pairs are correctly labeled

**3. Evaluation Settings:**
- **Increase evaluation samples**: Use `num_samples=100` or more for more reliable metrics
- **Test on different data**: Evaluate on a separate validation set, not training data

### Success Indicators

Your fine-tuning is successful when you see **positive changes** like:
- Coherence: +0.5 or higher
- Fluency: +0.3 or higher  
- Groundedness: +0.4 or higher

Iterate on hyperparameters and training data until your fine-tuned model consistently outperforms the base model!

## 14. Continual Fine-Tuning (Optional)

If your fine-tuned model didn't show improvements, you can perform **continual fine-tuning** by using the fine-tuned model as the base for another round of training. This iterative approach can help refine the model further.

### When to Use Continual Fine-Tuning:
- Your first fine-tuning run didn't improve metrics
- You want to adjust hyperparameters and train further
- You have additional training data to incorporate
- You need to fine-tune for a more specific task

### How It Works:
Instead of using model_name (base model), use fine_tuned_model_id from section 10 as your new base model. The code below is the same as section 8, but modified to continue training from your fine-tuned model.

In [16]:
# Continual fine-tuning is intentionally opt-in to avoid creating an unreviewed second billable job.
continual_job = None
print("Continual fine-tuning not requested; no retry job created.")


## 15. Next Steps

Congratulations! You've successfully fine-tuned a model with DPO.

### What's Next?
- Deploy your model to production
- Evaluate on more test cases
- Experiment with hyperparameters
- Try different datasets